# Quantitative Syntactic Analysis of the *Confessio Patricii*

**Corpus:** `confessio.conllu` (UDPipe 2, model `latin-ittb-ud-2.17`)
**Goal:** turn the manual syntactic observations recorded in Arborator Grew
into reproducible, corpus-wide quantitative evidence.

This notebook follows the methodology from the course lectures on
linguistic analysis (lecture 7) and stylistic analysis (lecture 9):
CoNLL-U as the data format, `deprel` distributions as measurable style
indicators, and explicit treebank guidelines (`guidelines_latin.pdf`,
based on the Prague Dependency Treebank) as the annotation standard
against which specific constructions are checked.

The notebook is organised in ten stages, each with a short explanation
before the code.

## Stage 1 -- Load the CoNLL-U File

The corpus is read with the `conllu` library, which parses the file into
a list of `TokenList` objects -- one per sentence, each with a
`.metadata` dict (sentence id, raw text) and a list of token dictionaries
(`id`, `form`, `lemma`, `upos`, `feats`, `head`, `deprel`, ...).

No filtering happens yet -- this stage only loads the raw data.

In [1]:
import io
from collections import Counter, defaultdict

from conllu import parse_incr

CONLLU_PATH = "confessio.conllu"

with io.open(CONLLU_PATH, "r", encoding="utf-8") as f:
    sentences = list(parse_incr(f))

print(f"Loaded {len(sentences)} sentences from {CONLLU_PATH}")

Loaded 13 sentences from confessio.conllu


## Stage 2 -- Helper Functions for Tree Metrics

Five helper functions do the actual dependency-tree work. Each one
answers a specific question that was first asked manually in the
Arborator Grew notes, and now gets a precise, repeatable definition:

- `build_children_map` -- turns the flat token list into a
  `{head_id: [children]}` map, the basic structure needed to walk a tree.
- `tree_depth` -- recursively computes how many levels deep a sentence's
  dependency tree goes (the "4 levels of embedding" noted by hand for
  sentence 9 becomes a computed number here).
- `longest_conj_chain` -- counts how many verbs are coordinated in a row
  under the same ultimate head (the 6-verb chain under `induxit` in
  sentence 5).
- `get_root_token` -- returns the token whose `head == 0`, i.e. the
  sentence's ROOT (relevant for the `est` existential-vs-copular
  distinction discussed in the notes).
- `branching_direction` -- counts how many dependents precede vs follow
  their head, a standard word-order measure.

Multiword tokens (ranges like `4-5`, which `conllu` keeps as non-integer
ids) are skipped everywhere, since they are not real nodes of the
dependency tree.

In [2]:
def build_children_map(tokens):
    """
    Build a mapping {head_id: [child_token, ...]} for one sentence.
    Multi-word tokens (ranges like '4-5') are skipped, since they
    are not part of the dependency tree itself.
    """
    children = defaultdict(list)
    for tok in tokens:
        # conllu library marks multiword/empty nodes with non-int ids
        if not isinstance(tok["id"], int):
            continue
        children[tok["head"]].append(tok)
    return children


def tree_depth(tokens):
    """
    Compute the maximum depth of the dependency tree for one sentence.
    Depth of the root is 0; depth of a child is depth(head) + 1.
    Returns the maximum depth found in the sentence.
    """
    children = build_children_map(tokens)

    def depth_of(token_id, current_depth):
        max_d = current_depth
        for child in children.get(token_id, []):
            max_d = max(max_d, depth_of(child["id"], current_depth + 1))
        return max_d

    # root token has head == 0
    root_tokens = [t for t in tokens if isinstance(t["id"], int) and t["head"] == 0]
    if not root_tokens:
        return 0
    return max(depth_of(t["id"], 0) for t in root_tokens)


def longest_conj_chain(tokens):
    """
    Find the longest chain of consecutive 'conj' relations sharing
    the same ultimate head (i.e. how many tokens are coordinated
    together in a single coordination chain, root included).

    Example: induxit (root) -> dispersit (conj) -> aperuit (conj) -> ...
    counts as a chain of length 6 (root + 5 conj siblings).
    """
    children = build_children_map(tokens)
    best = 0
    for head_id, kids in children.items():
        conj_kids = [k for k in kids if k["deprel"] == "conj"]
        if conj_kids:
            # chain length = 1 (the head itself) + number of conj children
            # but conj can also stack on top of conj, so walk recursively
            def count_conj_chain(tid):
                total = 0
                for k in children.get(tid, []):
                    if k["deprel"] == "conj":
                        total += 1 + count_conj_chain(k["id"])
                return total

            chain_len = 1 + count_conj_chain(head_id)
            best = max(best, chain_len)
    return best


def get_root_token(tokens):
    """Return the root token of the sentence (head == 0), or None."""
    for tok in tokens:
        if isinstance(tok["id"], int) and tok["head"] == 0:
            return tok
    return None


def branching_direction(tokens):
    """
    Count left-branching vs right-branching dependents.
    A dependent is 'left' if its id < head id (it precedes its head),
    'right' if it follows its head. Root token is excluded.
    Returns (left_count, right_count).
    """
    left, right = 0, 0
    for tok in tokens:
        if not isinstance(tok["id"], int):
            continue
        if tok["head"] == 0:
            continue
        if tok["id"] < tok["head"]:
            left += 1
        else:
            right += 1
    return left, right

print("Helper functions defined: build_children_map, tree_depth, "
      "longest_conj_chain, get_root_token, branching_direction")

Helper functions defined: build_children_map, tree_depth, longest_conj_chain, get_root_token, branching_direction


## Stage 3 -- Detecting Two Specific Late-Latin Constructions

Two functions target constructions already identified by hand in the
notes, in order to verify them mechanically across the whole corpus
rather than sentence by sentence:

1. **`find_modal_xcomp`** -- looks for modal/catenative verbs (`possum`,
   `volo`, `incipio`, and related verbs with the same syntactic
   behaviour) whose infinitival complement is attached as `xcomp`. This
   directly applies the rule quoted from `guidelines_latin.pdf` (3.3.6):
   *"verbs like possum, volo or incipio"* take an infinitive that
   behaves like an object, with the finite modal verb as the head.

2. **`find_ut_infinitive`** -- looks for `ut` (`deprel = mark`) governing
   a verb marked `VerbForm=Inf` instead of a finite subjunctive. This is
   the late-Latin / biblical-Latin feature the notes flagged for
   sentence 8 (`ut exaltare et confiteri` instead of classical
   `ut exaltemus`).

In [3]:
# Modal / semi-auxiliary verbs whose infinitival complement is OBJ/xcomp
# according to guidelines_latin.pdf (3.3.6): possum, volo, incipio.
# We extend lemma matching slightly to cover related forms (volo/nolo/malo
# share the same syntactic behaviour as catenative verbs in Latin).
MODAL_LEMMAS = {"possum", "volo", "incipio", "nolo", "malo", "queo", "debeo"}


def find_modal_xcomp(tokens):
    """
    Find pairs (modal_verb, infinitive) where an xcomp dependent of a
    modal/catenative verb is itself a non-finite verb (VerbForm=Inf).
    Returns a list of (modal_lemma, infinitive_form) tuples.
    """
    by_id = {t["id"]: t for t in tokens if isinstance(t["id"], int)}
    results = []
    for tok in tokens:
        if not isinstance(tok["id"], int):
            continue
        if tok["deprel"] == "xcomp":
            head = by_id.get(tok["head"])
            if head and head.get("lemma", "").lower() in MODAL_LEMMAS:
                feats = tok.get("feats") or {}
                if feats.get("VerbForm") == "Inf":
                    results.append((head["lemma"], tok["form"]))
    return results


def find_ut_infinitive(tokens):
    """
    Find cases of 'ut' (mark) governing a clause whose predicate is an
    infinitive (VerbForm=Inf) rather than a finite subjunctive verb.
    This is the late-Latin / biblical-Latin feature noted in the user's
    annotation notes (sentence 5: 'ut exaltare et confiteri').

    Returns a list of (ut_token_form, governed_verb_form) tuples.
    """
    by_id = {t["id"]: t for t in tokens if isinstance(t["id"], int)}
    results = []
    for tok in tokens:
        if not isinstance(tok["id"], int):
            continue
        if tok["form"].lower() == "ut" and tok["deprel"] == "mark":
            governed = by_id.get(tok["head"])
            if governed:
                feats = governed.get("feats") or {}
                if feats.get("VerbForm") == "Inf":
                    results.append((tok["form"], governed["form"]))
    return results

print("Construction detectors defined: find_modal_xcomp, find_ut_infinitive")

Construction detectors defined: find_modal_xcomp, find_ut_infinitive


## Stage 4 -- Computing Per-Sentence Metrics

This stage runs every helper function over every sentence and collects
the results into `per_sentence_rows` (a list of dicts, one per
sentence), plus three corpus-wide accumulators:

- `all_deprels` -- frequency count of every dependency relation label
  across the whole corpus
- `all_root_upos` -- frequency count of the part-of-speech of each
  sentence's ROOT
- `modal_xcomp_hits` / `ut_inf_hits` -- every occurrence found by the
  Stage 3 detectors, collected across all 13 sentences

This is the only loop that touches every sentence; everything after
this stage just reports on what was collected here.

In [4]:
per_sentence_rows = []
all_deprels = Counter()
all_root_upos = Counter()
modal_xcomp_hits = []
ut_inf_hits = []

for sent in sentences:
    tokens = [t for t in sent if isinstance(t["id"], int)]
    sent_id = sent.metadata.get("sent_id", "?")
    n_tokens = len(tokens)
    depth = tree_depth(tokens)
    conj_chain = longest_conj_chain(tokens)
    root_tok = get_root_token(tokens)
    root_upos = root_tok["upos"] if root_tok else "NA"
    root_lemma = root_tok["lemma"] if root_tok else "NA"
    left, right = branching_direction(tokens)

    per_sentence_rows.append({
        "sent_id": sent_id,
        "n_tokens": n_tokens,
        "tree_depth": depth,
        "longest_conj_chain": conj_chain,
        "root_lemma": root_lemma,
        "root_upos": root_upos,
        "left_deps": left,
        "right_deps": right,
    })

    for tok in tokens:
        all_deprels[tok["deprel"]] += 1
    if root_tok:
        all_root_upos[root_upos] += 1

    modal_xcomp_hits.extend(find_modal_xcomp(tokens))
    ut_inf_hits.extend(find_ut_infinitive(tokens))

print(f"Computed metrics for {len(per_sentence_rows)} sentences.")

Computed metrics for 13 sentences.


## Stage 5 -- Per-Sentence Table

A plain-text table, one row per sentence, showing token count, tree
depth, longest coordination chain, the ROOT's lemma and part of speech,
and the left/right branching split. This is the most direct, sentence-
level view of the data -- useful for spotting outliers like sentence 5
(deep tree, long conj-chain) or sentence 9 (extremely long, but
shallower relative to its length).

In [5]:
print("=" * 100)
print("PER-SENTENCE METRICS")
print("=" * 100)
col_id, col_tok, col_depth = "ID", "tokens", "depth"
col_conj, col_lemma, col_upos = "conj_chain", "root_lemma", "root_upos"
col_left, col_right = "left", "right"
header = (
    f"{col_id:>3} {col_tok:>7} {col_depth:>6} {col_conj:>11} "
    f"{col_lemma:>12} {col_upos:>9} {col_left:>5} {col_right:>6}"
)
print(header)
print("-" * len(header))
for row in per_sentence_rows:
    sid = row["sent_id"]
    n_tok = row["n_tokens"]
    depth_v = row["tree_depth"]
    conj_v = row["longest_conj_chain"]
    lemma_v = row["root_lemma"]
    upos_v = row["root_upos"]
    left_v = row["left_deps"]
    right_v = row["right_deps"]
    print(
        f"{sid:>3} {n_tok:>7} {depth_v:>6} {conj_v:>11} "
        f"{lemma_v:>12} {upos_v:>9} {left_v:>5} {right_v:>6}"
    )

PER-SENTENCE METRICS
 ID  tokens  depth  conj_chain   root_lemma root_upos  left  right
------------------------------------------------------------------
  1      27      5           3        habeo      VERB    13     13
  2      10      2           0        habeo      VERB     7      2
  3       6      1           0      sedecim       NUM     4      1
  4      41      6           3       ignoro      VERB    25     15
  5      90     11           5       induco      VERB    40     49
  6      26      7           2       possum      VERB    15     10
  7      26      5           2          hic       DET    13     12
  8      34      8           2        alius       DET    20     13
  9     132      6           4           do      VERB    70     61
 10       6      2           0         dico      VERB     3      2
 11      13      3           3       inuoco      VERB     3      9
 12       4      1           0       inquam      VERB     2      1
 13       9      3           2  honorific

## Stage 6 -- Corpus-Level Summary Statistics

Averages and extremes computed over `per_sentence_rows`: mean sentence
length, the single longest sentence, mean and maximum tree depth, mean
and maximum coordination-chain length, and the overall left/right
branching ratio for the whole corpus.

These numbers are the quantitative counterpart to the qualitative claim
in the notes that Patrick's *Confessio* is built from long, deeply
nested, heavily coordinated periods rather than short classical
sentences.

In [6]:
n_sent = len(per_sentence_rows)
total_tokens = sum(r["n_tokens"] for r in per_sentence_rows)
avg_len = total_tokens / n_sent
max_len_row = max(per_sentence_rows, key=lambda r: r["n_tokens"])
avg_depth = sum(r["tree_depth"] for r in per_sentence_rows) / n_sent
max_depth_row = max(per_sentence_rows, key=lambda r: r["tree_depth"])
avg_conj = sum(r["longest_conj_chain"] for r in per_sentence_rows) / n_sent
max_conj_row = max(per_sentence_rows, key=lambda r: r["longest_conj_chain"])

total_left = sum(r["left_deps"] for r in per_sentence_rows)
total_right = sum(r["right_deps"] for r in per_sentence_rows)

print("=" * 100)
print("CORPUS-LEVEL SUMMARY")
print("=" * 100)
print(f"Sentences: {n_sent}")
print(f"Total tokens (excluding multiword expansions): {total_tokens}")
print(f"Average sentence length: {avg_len:.1f} tokens")

longest_id = max_len_row["sent_id"]
longest_n = max_len_row["n_tokens"]
print(f"Longest sentence: #{longest_id} ({longest_n} tokens)")

print(f"Average tree depth: {avg_depth:.2f}")
deepest_id = max_depth_row["sent_id"]
deepest_d = max_depth_row["tree_depth"]
print(f"Deepest tree: #{deepest_id} (depth = {deepest_d})")

print(f"Average longest conj-chain per sentence: {avg_conj:.2f}")
conj_id = max_conj_row["sent_id"]
conj_len = max_conj_row["longest_conj_chain"]
conj_lemma = max_conj_row["root_lemma"]
print(f"Longest conj-chain overall: #{conj_id} (chain = {conj_len}, "
      f"root lemma = {conj_lemma})")

left_pct = total_left / (total_left + total_right)
right_pct = total_right / (total_left + total_right)
print(f"Branching direction (corpus-wide): "
      f"left = {total_left} ({left_pct:.1%}), "
      f"right = {total_right} ({right_pct:.1%})")

CORPUS-LEVEL SUMMARY
Sentences: 13
Total tokens (excluding multiword expansions): 424
Average sentence length: 32.6 tokens
Longest sentence: #9 (132 tokens)
Average tree depth: 4.62
Deepest tree: #5 (depth = 11)
Average longest conj-chain per sentence: 2.00
Longest conj-chain overall: #5 (chain = 5, root lemma = induco)
Branching direction (corpus-wide): left = 219 (53.3%), right = 192 (46.7%)


## Stage 7 -- Deprel Frequency Distribution

The full frequency table of every `deprel` label in the corpus, sorted
by count, exactly like the CoNLL-U queries described in lecture 9
(`upos=VERB & Tense=Imp & deprel=root` style queries -- here applied as a
full distribution rather than a single filter).

Two relevant labels are then grouped for interpretability:
subordination-related relations (`advcl`, `acl`, `acl:relcl`, `ccomp`,
`csubj`, `xcomp`) versus coordination-related relations (`conj`, `cc`).
The ratio between these two groups is the main quantitative signal for
the chained-coordination style noted by hand throughout the Arborator
Grew annotation.

In [7]:
print("=" * 100)
print("DEPREL FREQUENCY DISTRIBUTION (whole corpus)")
print("=" * 100)
col_deprel, col_count, col_pct = "deprel", "count", "% of tokens"
print(f"{col_deprel:>15} {col_count:>6} {col_pct:>12}")
print("-" * 36)
for deprel, count in all_deprels.most_common():
    pct = count / total_tokens
    print(f"{deprel:>15} {count:>6} {pct:>11.1%}")

# Key subordination-related deprels, grouped for interpretability
subordination_deprels = {"advcl", "acl", "acl:relcl", "ccomp", "csubj", "xcomp"}
sub_count = sum(all_deprels[d] for d in subordination_deprels if d in all_deprels)
sub_pct = sub_count / total_tokens
print()
print(f"Subordination-related deprels combined "
      f"(advcl, acl, acl:relcl, ccomp, csubj, xcomp): "
      f"{sub_count} tokens ({sub_pct:.1%} of corpus)")

coordination_count = all_deprels.get("conj", 0) + all_deprels.get("cc", 0)
coord_pct = coordination_count / total_tokens
print(f"Coordination-related deprels combined (conj, cc): "
      f"{coordination_count} tokens ({coord_pct:.1%} of corpus)")

DEPREL FREQUENCY DISTRIBUTION (whole corpus)
         deprel  count  % of tokens
------------------------------------
             cc     44       10.4%
          punct     43       10.1%
           conj     40        9.4%
           case     33        7.8%
            det     30        7.1%
            obj     29        6.8%
           nmod     25        5.9%
            obl     25        5.9%
          nsubj     20        4.7%
           mark     15        3.5%
           root     13        3.1%
        obl:arg     12        2.8%
            cop     11        2.6%
      acl:relcl     11        2.6%
          xcomp      9        2.1%
         advmod      9        2.1%
           amod      8        1.9%
          advcl      7        1.7%
      discourse      6        1.4%
    advmod:tmod      5        1.2%
       aux:pass      5        1.2%
            acl      4        0.9%
     advmod:neg      4        0.9%
     nsubj:pass      4        0.9%
           flat      4        0.9%
    adv

## Stage 8 -- ROOT Part-of-Speech Distribution

Counts how often each part of speech occupies the ROOT position across
the 13 sentences. This is directly relevant to the existential-vs-
copular `est` discussion in the notes for sentence 9: a copular `est`
is never ROOT (the predicate nominal or the subject takes that role
instead), while an existential `est` is ROOT of its own clause. A high
proportion of `VERB` roots, with occasional `DET`, `NUM`, or `ADJ` roots,
is exactly the pattern that distinction predicts.

In [8]:
print("=" * 100)
print("ROOT UPOS DISTRIBUTION")
print("=" * 100)
col_upos, col_count, col_pct = "upos", "count", "% of sentences"
print(f"{col_upos:>10} {col_count:>6} {col_pct:>15}")
print("-" * 34)
for upos, count in all_root_upos.most_common():
    pct = count / n_sent
    print(f"{upos:>10} {count:>6} {pct:>14.1%}")

ROOT UPOS DISTRIBUTION
      upos  count  % of sentences
----------------------------------
      VERB      9          69.2%
       DET      2          15.4%
       NUM      1           7.7%
       ADJ      1           7.7%


## Stage 9 -- The Two Targeted Constructions, Listed

The actual hits found by `find_modal_xcomp` and `find_ut_infinitive`
(Stage 3), printed individually. For a 13-sentence corpus these lists
are short by design -- the point is not raw frequency, but confirming
that the constructions identified by hand in Arborator Grew are also
detectable mechanically, and pinned to an explicit rule
(`guidelines_latin.pdf` 3.3.6 for the modal+infinitive case).

In [9]:
print("=" * 100)
print("MODAL/CATENATIVE VERB + INFINITIVE (xcomp) CONSTRUCTIONS")
print("(per guidelines_latin.pdf 3.3.6: possum, volo, incipio govern xcomp)")
print("=" * 100)
if modal_xcomp_hits:
    for modal_lemma, inf_form in modal_xcomp_hits:
        print(f"  {modal_lemma}  ->  xcomp({inf_form})")
else:
    print("  None found in this corpus.")
print(f"Total occurrences: {len(modal_xcomp_hits)}")

print()
print("=" * 100)
ut_title = "UT" + " + INFINITIVE CONSTRUCTIONS (late-Latin / biblical-Latin feature)"
print(ut_title)
ut_subtitle = "(classical Latin would expect " + "UT" + " + subjunctive)"
print(ut_subtitle)
print("=" * 100)
if ut_inf_hits:
    for ut_form, verb_form in ut_inf_hits:
        print(f"  {ut_form} ... {verb_form} (infinitive)")
else:
    print("  None found in this corpus.")
print(f"Total occurrences: {len(ut_inf_hits)}")

MODAL/CATENATIVE VERB + INFINITIVE (xcomp) CONSTRUCTIONS
(per guidelines_latin.pdf 3.3.6: possum, volo, incipio govern xcomp)
  possum  ->  xcomp(tacere)
Total occurrences: 1

UT + INFINITIVE CONSTRUCTIONS (late-Latin / biblical-Latin feature)
(classical Latin would expect UT + subjunctive)
  ut ... exaltare (infinitive)
Total occurrences: 1


## Stage 10 -- Known Discrepancy: Auto-Parse vs Manual Annotation

This stage is not a bug fix -- it is an explicit record of a place where
the UDPipe automatic parse and the manual Arborator Grew annotation
disagree, kept visible rather than silently resolved.

The manual notes for sentence 9 analyse *"quem ... fuisse testamur"* as
an Accusativus cum Infinitivo, with `fuisse` as `ccomp` of `testamur`.
The UDPipe parse in this file instead attaches `fuisse` to `Patre` via
`acl:relcl`, governed by `testamur` -- and there is in fact **no `ccomp`
label anywhere in this corpus** (confirmed by the Stage 7 frequency
table above). Resolving this requires a manual treebank edit in
Arborator Grew, not a scripting change, which is exactly why it is
reported here as a finding rather than corrected silently.

In [10]:
print("=" * 100)
print("KNOWN DISCREPANCY: AUTO-PARSE vs MANUAL ANNOTATION (sentence 9)")
print("=" * 100)
print("Manual notes (Arborator Grew) analyse the form fuisse as ccomp(testamur)")
print("-> Accusativus cum Infinitivo reading.")
print("UDPipe auto-parse instead attaches fuisse to Patre via")
print("acl:relcl, governed by testamur. No ccomp deprel occurs")
print("anywhere in this corpus. This divergence is a candidate for")
print("manual correction in Arborator Grew, not a scripting artifact.")

KNOWN DISCREPANCY: AUTO-PARSE vs MANUAL ANNOTATION (sentence 9)
Manual notes (Arborator Grew) analyse the form fuisse as ccomp(testamur)
-> Accusativus cum Infinitivo reading.
UDPipe auto-parse instead attaches fuisse to Patre via
acl:relcl, governed by testamur. No ccomp deprel occurs
anywhere in this corpus. This divergence is a candidate for
manual correction in Arborator Grew, not a scripting artifact.


## Summary

The notebook turns ten qualitative observations from the Arborator Grew
annotation notes into ten reproducible quantitative stages: tree depth,
coordination-chain length, deprel distribution, ROOT part-of-speech
distribution, two targeted late-Latin constructions, and one explicitly
flagged annotation discrepancy. Every number above can be regenerated
from `confessio.conllu` alone, with no manual recomputation required --
which is the reproducibility standard the course lectures set for
quantitative philological evidence.